# Many Models Forecasting Demo

This notebook showcases how to run MMF with MLForecast-based global models on multiple time series of monthly resolution. We use [`MLForecastLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (LightGBM with fixed hyperparameters) and [`MLForecastAutoLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (Optuna-tuned LightGBM + feature pipeline), both built on top of [MLForecast](https://nixtlaverse.nixtla.io/mlforecast/index.html). We use [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5) monthly data.

Unlike the [`global_monthly_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/monthly/global_monthly_dl.ipynb) notebook (GPU-bound neural forecasters), this notebook stays on CPU and runs end-to-end in a single Python process.

### Cluster setup

We recommend using a **single-node CPU cluster** with [Databricks Runtime 17.3 LTS for ML](https://docs.databricks.com/en/release-notes/runtime/17.3lts-ml.html). The pinned versions in [`requirements-global.txt`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) (`mlforecast==1.0.31`, `lightgbm==4.6.0`, `optuna==3.6.1`) match the DBR 17.3 LTS ML preinstalled versions exactly.

### Install and import packages

In [ ]:
%pip install -r ../../requirements-global.txt --quiet
%pip install datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

In [ ]:
import logging
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4
from mmf_sa import run_forecast

logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Prepare data
Using [`datasetsforecast`](https://github.com/Nixtla/datasetsforecast/tree/main/) to download M4 monthly data.

In [ ]:
# Number of time series
n = 100


def create_m4_monthly():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Monthly")
    target_ids = {f"M{i}" for i in range(1, n)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 60:
        df = df.iloc[-60:]
    start = pd.Timestamp("2018-01-01")
    date_idx = pd.date_range(start=start, periods=len(df), freq="ME", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

In [ ]:
catalog = "mmf"
db = "m4"
user = spark.sql('select current_user() as user').collect()[0]['user']

In [ ]:
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_monthly())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.m4_monthly_train")
)

In [ ]:
display(
  spark.sql(f"select * from {catalog}.{db}.m4_monthly_train where unique_id in ('M1', 'M2', 'M3', 'M4', 'M5') order by unique_id, ds")
  )

Note that monthly forecasting requires the timestamp column to represent the last day of each month (`pd.date_range(..., freq="ME")` enforces this above).

### Models

Both models are configured in [`mmf_sa/models/models_conf.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/models_conf.yaml) with monthly-frequency-specific defaults in [`mmf_sa/forecasting_conf_monthly.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/forecasting_conf_monthly.yaml). The monthly defaults use lags `[1, 3, 6, 12]` (last month, quarter, half-year, year) and date features `month` + `quarter`.

In [ ]:
active_models = [
    "MLForecastLGBM",
    "MLForecastAutoLGBM",
]

### Run MMF

In [ ]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.m4_monthly_train",
    scoring_data=f"{catalog}.{db}.m4_monthly_train",
    scoring_output=f"{catalog}.{db}.monthly_scoring_output",
    evaluation_output=f"{catalog}.{db}.monthly_evaluation_output",
    model_output=f"{catalog}.{db}",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="M",
    prediction_length=3,
    backtest_length=12,
    stride=1,
    metric="smape",
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m4_monthly",
    use_case_name="m4_monthly",
    accelerator="cpu",
)

### Evaluate

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.monthly_evaluation_output
    where unique_id = 'M1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, backtest_window_start_date
    """))

### Forecast

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.monthly_scoring_output
    where unique_id = 'M1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, ds
    """))

### Delete Tables

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.monthly_evaluation_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.monthly_scoring_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))